# Sampling origins across different cities and creating origin-destination pairs.

## Prerequisites
This repository builds on the python package OSMNx (v.2.0.1, https://osmnx.readthedocs.io/en/stable/). I recommend installing it via conda:
```
conda create -n ox -c conda-forge --strict-channel-priority osmnx
```
For sampling nodes based on city names two additional packages are required, namely geopy (v.2.3.1, https://geopy.readthedocs.io/en/stable/) and overpy (v.0.7, https://python-overpy.readthedocs.io/en/latest/)

```
pip install geopy
pip install overpy nodes run Ubuntu Jammy 22.04 LTS.
There is local scratch space on each node, which is shared between the jobs currently running. Connected to Kebnekaise is also our parallel file system Ransarn (where your project storage is located), which provide quick access to files regardless of which node they run on. For more information about the different file systems that are available on our systems, read the Filesystems and Storage page.
```

For visualizing routes and geometry on maps I use the folium package (v.0.19.4, https://python-visualization.github.io/folium/latest/) that is included in the OSMNx package, but for creating static images of these visualizations the Selenium package is required (v.4.28.0, https://www.selenium.dev/documentation/)

```
pip install selenium
```

## This exampleCities are used as the basis to find random samples of intersections. The region and country names are nice to have, but they are not necessary.

In [1]:
# region <set up parameters>

import os
import csv
import pandas as pd
import multiprocessing

from datetime import datetime
sample_size = 5
min_distance = 2
random_seed = 4
network_type = 'drive'
point_distance_size = 10000
experiment_name = "2025-11-illustration"
#base_path=f"/proj/nobackup/streetnetwork-alignment/{experiment_name}" for HPC2N
base_path=f"/media/arvidkh/projekt/streetnetwork-alignment/{experiment_name}"
min_od_distance = 4750
max_od_distance = 5250
od_pair_sample_size = 180


if not os.path.exists(base_path):
    os.makedirs(base_path)

parameters_file_path = os.path.join(base_path, f"parameters_2025_12_01.csv")
city_sample_nodes_path = os.path.join(base_path, f'city_sample_nodes_{experiment_name}.csv')
local_graph_folder = os.path.join(base_path, 'local_origin_graphs')

with open(parameters_file_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(["Parameter", "Value"])
    writer.writerow(["sample_size", sample_size])
    writer.writerow(["min_distance", min_distance])
    writer.writerow(["random_seed", random_seed])
    writer.writerow(["network_type", network_type])
    writer.writerow(["point_distance_size", point_distance_size])
    writer.writerow(["min_od_distance", min_od_distance])
    writer.writerow(["max_od_distance", max_od_distance])
    writer.writerow(["base_path", base_path])
    writer.writerow(["city_sample_nodes_path", city_sample_nodes_path])
    writer.writerow(["local_graph_folder", local_graph_folder])
    writer.writerow(["base_path", base_path])
    writer.writerow(["experiment_name", experiment_name])


param = pd.read_csv(parameters_file_path)
display(param)



num_processes = multiprocessing.cpu_count()  # Adjust based on your system's capabilities
print(f"Number of processes to use: {num_processes}")

# endregion

,Parameter,Value
0,sample_size,5
1,min_distance,2
2,random_seed,4
3,network_type,drive
4,point_distance_size,10000
5,min_od_distance,4750
6,max_od_distance,5250
7,base_path,/media/arvidkh/projekt/streetnetwork-alignment...
8,city_sample_nodes_path,/media/arvidkh/projekt/streetnetwork-alignment...
9,local_graph_folder,/media/arvidkh/projekt/streetnetwork-alignment...


Number of processes to use: 8


In [2]:
# region <get node sample from cities>
# The workflow for analyzing the routes begins with coordinate points used as origin locations.
import os 
import pandas as pd
from route_network_analysis import node_sampling
df = pd.read_csv("16_city_sample.csv")

city_sample_nodes_path = os.path.join(base_path, f'city_sample_nodes_{experiment_name}.csv')

if os.path.exists(city_sample_nodes_path):
    print("sample nodes already retrieved")
else:
    node_sample_df = node_sampling.get_random_nodes_for_all_cities(df, min_distance_km=min_distance, sample_size=sample_size,random_seed=random_seed)
    node_sample_df['graph_path'] = node_sample_df.apply(lambda row: os.path.join(local_graph_folder, f"{row['city_name_en']}_{row['node_id']}.graphml"), axis=1)
    node_sample_df.to_csv(os.path.join(base_path, city_sample_nodes_path))

# endregion

Random nodes for Lima that are at least 2 km apart: [1330495156, 1359116964, 848394908, 11050450864, 3107812023]
Random nodes for Lima: [1330495156, 1359116964, 848394908, 11050450864, 3107812023]
Node: 1330495156
Random node coordinates for Lima: -11.9789855, -77.0547884
Node: 1359116964
Random node coordinates for Lima: -12.0097364, -77.0931582
Node: 848394908
Random node coordinates for Lima: -12.0276739, -77.0958306
Node: 11050450864
Random node coordinates for Lima: -11.9433227, -77.0076102
Node: 3107812023
Random node coordinates for Lima: -11.8927561, -76.9439907
Random nodes for Buenos Aires that are at least 2 km apart: [1406097496, 1422501545, 1357237840, 5533758329, 9852258020]
Random nodes for Buenos Aires: [1406097496, 1422501545, 1357237840, 5533758329, 9852258020]
Node: 1406097496
Random node coordinates for Buenos Aires: -33.3550713, -60.2680864
Node: 1422501545
Random node coordinates for Buenos Aires: -33.2935262, -60.2644707
Node: 1357237840
Random node coordinates f

In [3]:
# region <create graphs>
import os  # for file operations
import pandas as pd  # for reading the csv file
import joblib
import logging
import ast
import route_network_analysis as rna
import osmnx as ox
logging.basicConfig(level=logging.ERROR, format='%(asctime)s - %(levelname)s - %(message)s',filename='jupyter.log', filemode='w')

#
sub_folder = "local_origin_graphs"
local_graph_folder = os.path.join(base_path, sub_folder)
if not os.path.exists(local_graph_folder):
    os.makedirs(local_graph_folder)

print(city_sample_nodes_path)
df = pd.read_csv(city_sample_nodes_path)
print(df['graph_path'])
def create_graphs(row):
    if os.path.exists(row['graph_path']):
        print(f"Graph exists: {row['graph_path']}")
        return True, row['city_name'], row['node_id']
    else:
        print(f"---Graph missing: {row['graph_path']}---")
        # Apply the function asynchronously
    try:
        latlon_point = ast.literal_eval(row['node_latlon'])
        og = rna.origin_graph(origin_point=latlon_point, distance_from_point=point_distance_size,
                          city_name=row["city_name_en"], network_type=network_type, remove_parallel=True, simplify=True)
    
        og.save_graph(row['graph_path'])
    
        # Plot the origin graph to see if something is obviously wrong
        ox.plot_graph(og.graph, node_color='blue', node_size=5, edge_linewidth=1, edge_color='black', bgcolor='white',
                       save=True, filepath=os.path.join(local_graph_folder, f"{row['city_name']}_{row['node_id']}.png"), show=False)
        logging.error(f"Finished with graph: {row['graph_path']}")
        return True, row['city_name'], row['node_id']

    except Exception as e:
        logging.error(f"error {e} creating {row['graph_path']}")
        return False, row['city_name'], row['node_id'], e
        

# Number of processes to use
num_processes = (joblib.cpu_count()-2)
print(f"Number of processes to use: {num_processes}")

# Collect results from joblib
results = []
try:

    results = joblib.Parallel(n_jobs=num_processes,backend='loky')(
        joblib.delayed(create_graphs)(row) for _, row in df.iterrows()
    )
except Exception as e:
    print(f"Joblib parallel processing error: {e}")
    
for result in results:
    if not result[0]: 
        print(f"failed creating graph {result[1]}{result[2]}. error {result[3]}")
    else:
        print(f"finished creating graph {result[1]}{result[2]}")

print("finished")

# endregion

/media/arvidkh/projekt/streetnetwork-alignment/2025-11-illustration/city_sample_nodes_2025-11-illustration.csv
0     /media/arvidkh/projekt/streetnetwork-alignment...
1     /media/arvidkh/projekt/streetnetwork-alignment...
2     /media/arvidkh/projekt/streetnetwork-alignment...
3     /media/arvidkh/projekt/streetnetwork-alignment...
4     /media/arvidkh/projekt/streetnetwork-alignment...
                            ...                        
70    /media/arvidkh/projekt/streetnetwork-alignment...
71    /media/arvidkh/projekt/streetnetwork-alignment...
72    /media/arvidkh/projekt/streetnetwork-alignment...
73    /media/arvidkh/projekt/streetnetwork-alignment...
74    /media/arvidkh/projekt/streetnetwork-alignment...
Name: graph_path, Length: 75, dtype: object
Number of processes to use: 6
Graph exists: /media/arvidkh/projekt/streetnetwork-alignment/2025-11-illustration/local_origin_graphs/Lima_1330495156.graphml
Graph exists: /media/arvidkh/projekt/streetnetwork-alignment/2025-11-illu

In [12]:
# The next step is to add weights to the edges of the graph.
import pandas as pd # for reading the csv file
import joblib # replacing multiprocessing with joblib
df = pd.read_csv(city_sample_nodes_path)

def add_graph_weights(row):
    og = rna.origin_graph.from_graphml(graphml_path=row['graph_path'])
    og.add_simplest_paths_from_origin()
    og.add_weights('deviation_from_prototypical')
    og.add_weights('node_degree')
    og.add_weights('instruction_equivalent')
    #og.add_weights('betweenness_centrality')
    og.save_graph(row['graph_path'])
    print(f"Finished with graph: {row['city_name_en']} node: {row['node_id']}",flush=True)
    return True, row['city_name_en'], row['node_id']

# Number of processes to use
#num_processes = (joblib.cpu_count()-2)
#print(f"Number of processes to use: {num_processes}")
# Collect results from joblib
rows_to_process = []
for idx, row in df.iterrows():
    if row['weight_added'] == True:
        results.append(add_graph_weights(row))
    #rows_to_process.append(row)


#results = joblib.Parallel(n_jobs=num_processes, backend='loky')(
#    joblib.delayed(add_graph_weights)(row) for row in rows_to_process
#)


for result in results:
    if result[0]:
        mask = (df['city_name_en'] == result[1]) & (df['node_id'] == result[2])
        df.loc[mask, 'weights_added'] = True
df.to_csv(city_sample_nodes_path)

Finished with graph: Lima node: 1330495156
Finished with graph: Lima node: 1359116964
Finished with graph: Lima node: 848394908
Finished with graph: Lima node: 11050450864
Finished with graph: Lima node: 3107812023
Finished with graph: Buenos Aires node: 1406097496
Finished with graph: Buenos Aires node: 1422501545
Finished with graph: Buenos Aires node: 1357237840
Finished with graph: Buenos Aires node: 5533758329
Finished with graph: Buenos Aires node: 9852258020
Finished with graph: Mexico City node: 506391266
Finished with graph: Mexico City node: 762649538
Finished with graph: Mexico City node: 280151390
Finished with graph: Mexico City node: 7951676238
Finished with graph: Mexico City node: 2335775204
Finished with graph: Miami node: 99232717
Finished with graph: Miami node: 99343234
Finished with graph: Miami node: 99119377
Finished with graph: Miami node: 3505119075
Finished with graph: Miami node: 8073291354
Finished with graph: Pittsburgh node: 105097688
Finished with graph: 

In [ ]:
import pandas as pd # for reading the csv file
import joblib # replacing multiprocessing with joblib
import route_network_analysis as rna

df = pd.read_csv(city_sample_nodes_path)
display(df)
local_odpair_folder = os.path.join(base_path, "od_pair_data")
local_odpair_base_folder = os.path.join(local_odpair_folder, 'base')
local_odpair_geom_folder = os.path.join(local_odpair_folder, 'geom')
print(f"odpair data will be stored at {local_odpair_folder}")
os.makedirs(local_odpair_folder, exist_ok=True)
os.makedirs(local_odpair_base_folder, exist_ok=True)
os.makedirs(local_odpair_geom_folder, exist_ok=True)


df['od_pairs_added'] = False


def get_od_pairs(row):
    print(f"Finding OD_pairs for graph: {row['city_name_en']} node: {row['node_id']}",flush=True)
    og = rna.origin_graph.from_graphml(graphml_path=row['graph_path'])
    og.create_od_pairs(min_radius=min_od_distance, max_radius=max_od_distance, sample_size=od_pair_sample_size)
    od_pair_data = og.get_od_pair_data()
    od_pair_geom_data = og.get_od_pair_geom_data()
    json_base_path = os.path.join(local_odpair_base_folder, f"od_pair_{row['city_name_en']}_{row['node_id']}.json")
    json_geom_path = os.path.join(local_odpair_geom_folder, f"od_pair_geom_{row['city_name_en']}_{row['node_id']}.json")
    od_pair_data.to_json(json_base_path, orient="records", default_handler=str, indent=2)
    od_pair_geom_data.to_json(json_geom_path, orient="records", default_handler=str, indent=2)
    print(f"Finished finding OD_pairs for graph: {row['city_name_en']} node: {row['node_id']}",flush=True)
    return True,row['city_name_en'],row['node_id']


num_processes = (joblib.cpu_count() - 4)
print(f"Number of processes to use: {num_processes}")



rows_to_process = []
for idx, row in df.iterrows():
    #if not row['od_pairs_added']:
        rows_to_process.append(row)

results = joblib.Parallel(n_jobs=num_processes, backend='loky')(
    joblib.delayed(get_od_pairs)(row) for row in rows_to_process
)

for result in results:
    if result[0]:
        mask = (df['city_name_en'] == result[1]) & (df['node_id'] == result[2])
        df.loc[mask, 'od_pairs_added'] = True


import glob

# Get all json files from od_pair_data folder
od_pair_base_files = glob.glob(os.path.join(local_odpair_base_folder, "*.json"))
od_pair_geom_files = glob.glob(os.path.join(local_odpair_geom_folder, "*.json"))
# Read and combine all json files
od_pair_base_data = pd.concat([pd.read_json(f) for f in od_pair_base_files], ignore_index=False)
od_pair_geom_data = pd.concat([pd.read_json(f) for f in od_pair_geom_files], ignore_index=False)

print(f"Total number of od-pairs: {len(od_pair_base_data)}")
print(od_pair_base_data.columns)


# The od-pair data contains lists and dictionaries that are not easily saved to a csv file, so we store it as a json file.
# Still, there some columns that need to be serialized to strings such as shapely polygon objects.



od_pair_data_geom_path_json = os.path.join(local_odpair_folder, 'origin_od_pair_geom.json')
od_pair_data_base_path_csv = os.path.join(local_odpair_folder, 'origin_od_pair_base.csv')

od_pair_base_data.to_csv(od_pair_data_base_path_csv)
od_pair_geom_data.to_json(od_pair_data_geom_path_json, orient="records", default_handler=str, indent=2)

print("done")



,Unnamed: 0.1,Unnamed: 0,city_name,city_name_en,country_name,country_name_en,continent,region,network_type,node_id,node_latlon,graph_path,weights_added
0,0,0,Lima,Lima,Perú,Peru,South America,Latin America,drive,1330495156,"(-11.9789855, -77.0547884)",/media/arvidkh/projekt/streetnetwork-alignment...,True
1,1,1,Lima,Lima,Perú,Peru,South America,Latin America,drive,1359116964,"(-12.0097364, -77.0931582)",/media/arvidkh/projekt/streetnetwork-alignment...,True
2,2,2,Lima,Lima,Perú,Peru,South America,Latin America,drive,848394908,"(-12.0276739, -77.0958306)",/media/arvidkh/projekt/streetnetwork-alignment...,True
3,3,3,Lima,Lima,Perú,Peru,South America,Latin America,drive,11050450864,"(-11.9433227, -77.0076102)",/media/arvidkh/projekt/streetnetwork-alignment...,True
4,4,4,Lima,Lima,Perú,Peru,South America,Latin America,drive,3107812023,"(-11.8927561, -76.9439907)",/media/arvidkh/projekt/streetnetwork-alignment...,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,70,70,Shanghai,Shanghai,中国,China,Asia,Asia/Oceania,drive,622623197,"(31.2047956, 121.4662599)",/media/arvidkh/projekt/streetnetwork-alignment...,True
71,71,71,Shanghai,Shanghai,中国,China,Asia,Asia/Oceania,drive,838507201,"(31.1842744, 121.3899739)",/media/arvidkh/projekt/streetnetwork-alignment...,True
72,72,72,Shanghai,Shanghai,中国,China,Asia,Asia/Oceania,drive,475861592,"(31.2502287, 121.4892488)",/media/arvidkh/projekt/streetnetwork-alignment...,True
73,73,73,Shanghai,Shanghai,中国,China,Asia,Asia/Oceania,drive,11751956757,"(31.21574, 121.3964503)",/media/arvidkh/projekt/streetnetwork-alignment...,True


odpair data will be stored at /media/arvidkh/projekt/streetnetwork-alignment/2025-11-illustration/od_pair_data
Number of processes to use: 4


In [ ]:
import pandas as pd # for reading the csv file
import joblib # replacing multiprocessing with joblib
import route_network_analysis as rna

df = pd.read_csv(city_sample_nodes_path)
display(df)
local_odpair_folder = os.path.join(base_path, "od_pair_data")
local_odpair_base_folder = os.path.join(local_odpair_folder, 'base')
local_odpair_geom_folder = os.path.join(local_odpair_folder, 'geom')
print(f"odpair data will be stored at {local_odpair_folder}")
os.makedirs(local_odpair_folder, exist_ok=True)
os.makedirs(local_odpair_base_folder, exist_ok=True)
os.makedirs(local_odpair_geom_folder, exist_ok=True)


df['od_pairs_added'] = False


def get_od_pairs(row):
    print(f"Finding OD_pairs for graph: {row['city_name_en']} node: {row['node_id']}",flush=True)
    og = rna.origin_graph.from_graphml(graphml_path=row['graph_path'])
    og.create_od_pairs(min_radius=min_od_distance, max_radius=max_od_distance, sample_size=od_pair_sample_size)
    od_pair_data = og.get_od_pair_data()
    od_pair_geom_data = og.get_od_pair_geom_data()
    json_base_path = os.path.join(local_odpair_base_folder, f"od_pair_{row['city_name_en']}_{row['node_id']}.json")
    json_geom_path = os.path.join(local_odpair_geom_folder, f"od_pair_geom_{row['city_name_en']}_{row['node_id']}.json")
    od_pair_data.to_json(json_base_path, orient="records", default_handler=str, indent=2)
    od_pair_geom_data.to_json(json_geom_path, orient="records", default_handler=str, indent=2)
    print(f"Finished finding OD_pairs for graph: {row['city_name_en']} node: {row['node_id']}",flush=True)
    return True,row['city_name_en'],row['node_id']


num_processes = (joblib.cpu_count() - 4)
print(f"Number of processes to use: {num_processes}")



rows_to_process = []
for idx, row in df.iterrows():
        json_base_path = os.path.join(local_odpair_base_folder, f"od_pair_{row['city_name_en']}_{row['node_id']}.json")
        if not os.path.exists(json_base_path):
            rows_to_process.append(row)

results = joblib.Parallel(n_jobs=num_processes, backend='loky')(
    joblib.delayed(get_od_pairs)(row) for row in rows_to_process
)

for result in results:
    if result[0]:
        mask = (df['city_name_en'] == result[1]) & (df['node_id'] == result[2])
        df.loc[mask, 'od_pairs_added'] = True


import glob


# Get all json files from od_pair_data folder
od_pair_base_files = glob.glob(os.path.join(local_odpair_base_folder, "*.json"))
od_pair_geom_files = glob.glob(os.path.join(local_odpair_geom_folder, "*.json"))
# Read and combine all json files
od_pair_base_data = pd.concat([pd.read_json(f) for f in od_pair_base_files], ignore_index=False)
od_pair_geom_data = pd.concat([pd.read_json(f) for f in od_pair_geom_files], ignore_index=False)

print(f"Total number of od-pairs: {len(od_pair_base_data)}")
print(od_pair_base_data.columns)


# The od-pair data contains lists and dictionaries that are not easily saved to a csv file, so we store it as a json file.
# Still, there some columns that need to be serialized to strings such as shapely polygon objects.



od_pair_data_geom_path_json = os.path.join(local_odpair_folder, 'origin_od_pair_geom.json')
od_pair_data_base_path_csv = os.path.join(local_odpair_folder, 'origin_od_pair_base.csv')

od_pair_base_data.to_csv(od_pair_data_base_path_csv)
od_pair_geom_data.to_json(od_pair_data_geom_path_json, orient="records", default_handler=str, indent=2)

print("done")



In [2]:
import pandas as pd
import os
import glob

# Assuming base_path and city_sample_nodes_path are defined
df = pd.read_csv(city_sample_nodes_path)
display(df)

local_odpair_folder = os.path.join(base_path, "od_pair_data")
local_odpair_base_folder = os.path.join(local_odpair_folder, 'base')
local_odpair_geom_folder = os.path.join(local_odpair_folder, 'geom')
local_odpair_bearings_folder = os.path.join(local_odpair_folder, 'bearings')

print(f"odpair data will be stored at {local_odpair_folder}")
os.makedirs(local_odpair_folder, exist_ok=True)
os.makedirs(local_odpair_base_folder, exist_ok=True)
os.makedirs(local_odpair_geom_folder, exist_ok=True)
os.makedirs(local_odpair_bearings_folder, exist_ok=True)

# Get all json files from od_pair_data folder
od_pair_base_files = glob.glob(os.path.join(local_odpair_base_folder, "*.json"))
od_pair_geom_files = glob.glob(os.path.join(local_odpair_geom_folder, "*.json"))
od_pair_bearings_files = glob.glob(os.path.join(local_odpair_bearings_folder, "*.json"))

# Define bearing columns
bearing_columns = [
    "bearings_undirected",
    "bearings_undirected_weights",
    "bearings_directed",
    "bearings_directed_weights",
    "bearings_od_fwd",
    "bearings_od_bwd",
    "bearings_od_perp_fwd",
    "bearings_od_perp_bwd"
]

# Process each base JSON file to separate bearing data
for f in od_pair_base_files:
    data = pd.read_json(f)
    bearings_data = data[['id'] + bearing_columns]
    data.drop(columns=bearing_columns, inplace=True)

    # Save the processed base data back to the base folder
    base_filename = os.path.basename(f)
    data.to_json(os.path.join(local_odpair_base_folder, base_filename), orient="records", indent=2)

    # Save the bearings data to the bearings folder
    bearings_filename = base_filename.replace('.json', '_bearings.json')
    bearings_data.to_json(os.path.join(local_odpair_bearings_folder, bearings_filename), orient="records", indent=2)

# Concatenate geom data
#od_pair_geom_data = pd.concat([pd.read_json(f) for f in od_pair_geom_files], ignore_index=False)

print("Processing complete")

# Save the concatenated geom data
#od_pair_data_geom_path_json = os.path.join(local_odpair_folder, 'origin_od_pair_geom.json')
#od_pair_geom_data.to_json(od_pair_data_geom_path_json, orient="records", default_handler=str, indent=2)

print("done")


,Unnamed: 0.7,Unnamed: 0.6,Unnamed: 0.5,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,city_name,city_name_en,country_name,country_name_en,continent,region,network_type,node_id,node_latlon,graph_path,weights_added,od_pairs_added
0,0,0,0,0,0,0,0,0,Bangkok,Bangkok,ประเทศไทย,Thailand,Asia,Asia/Oceania,drive,2733298223,"(13.7253693, 100.4953615)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
1,1,1,1,1,1,1,1,1,Bangkok,Bangkok,ประเทศไทย,Thailand,Asia,Asia/Oceania,drive,3825324563,"(13.7843515, 100.4392777)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
2,2,2,2,2,2,2,2,2,Bangkok,Bangkok,ประเทศไทย,Thailand,Asia,Asia/Oceania,drive,1687458348,"(13.7918891, 100.5686954)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
3,3,3,3,3,3,3,3,3,Bangkok,Bangkok,ประเทศไทย,Thailand,Asia,Asia/Oceania,drive,7926838114,"(13.7020039, 100.5380657)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
4,4,4,4,4,4,4,4,4,Bangkok,Bangkok,ประเทศไทย,Thailand,Asia,Asia/Oceania,drive,4772368452,"(13.668279, 100.4044342)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,495,495,495,495,495,495,495,495,Washington,Washington,United States,United States,North America,US/Canada,drive,49896757,"(38.9773557, -77.0191765)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
496,496,496,496,496,496,496,496,496,Washington,Washington,United States,United States,North America,US/Canada,drive,50654959,"(38.8068877, -76.9933723)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
497,497,497,497,497,497,497,497,497,Washington,Washington,United States,United States,North America,US/Canada,drive,49761428,"(38.885008, -76.928403)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
498,498,498,498,498,498,498,498,498,Washington,Washington,United States,United States,North America,US/Canada,drive,11368228087,"(38.856723, -76.9699374)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True


odpair data will be stored at /proj/nobackup/streetnetwork-alignment/2025-04-full_dataset/od_pair_data
Processing complete
done


In [ ]:
import pandas as pd
import os
import glob
import numpy as np

# Assuming base_path and city_sample_nodes_path are defined
df = pd.read_csv(city_sample_nodes_path)
display(df)

local_odpair_folder = os.path.join(base_path, "od_pair_data")
local_odpair_base_folder = os.path.join(local_odpair_folder, 'base')
local_odpair_geom_folder = os.path.join(local_odpair_folder, 'geom')
local_odpair_bearings_folder = os.path.join(local_odpair_folder, 'bearings')

print(f"odpair data will be stored at {local_odpair_folder}")
os.makedirs(local_odpair_folder, exist_ok=True)
os.makedirs(local_odpair_base_folder, exist_ok=True)
os.makedirs(local_odpair_geom_folder, exist_ok=True)
os.makedirs(local_odpair_bearings_folder, exist_ok=True)

# Get all json files from od_pair_data folder

# 1. Gather all bearings JSON files
od_pair_bearings_files = glob.glob(os.path.join(local_odpair_bearings_folder, "*.json"))

# 2. Determine chunk sizes for splitting files into three groups
num_files = len(od_pair_bearings_files)
chunk_indices = np.array_split(np.arange(num_files), 3)

# 3. For each chunk, load only those files, concatenate, and save as a separate file
od_pair_bearings_chunk_paths = []
for i, indices in enumerate(chunk_indices):
    chunk_files = [od_pair_bearings_files[j] for j in indices]
    chunk_df = pd.concat([pd.read_json(f) for f in chunk_files], ignore_index=False)
    chunk_path = os.path.join(local_odpair_bearings_folder, f"od_pair_bearings_chunk_{i+1}.csv")
    chunk_df.to_csv(chunk_path)
    od_pair_bearings_chunk_paths.append(chunk_path)

# Optionally, load the three chunks as DataFrames (one at a time, if needed)
# od_pair_bearings_chunk1 = pd.read_csv(od_pair_bearings_chunk_paths[0])
# od_pair_bearings_chunk2 = pd.read_csv(od_pair_bearings_chunk_paths[1])
# od_pair_bearings_chunk3 = pd.read_csv(od_pair_bearings_chunk_paths[2])


od_pair_base_files = glob.glob(os.path.join(local_odpair_base_folder, "*.json"))
od_pair_base_data = pd.concat([pd.read_json(f) for f in od_pair_base_files], ignore_index=False)
od_pair_base_data.to_csv(od_pair_data_base_path_csv)


od_pair_geom_files = glob.glob(os.path.join(local_odpair_geom_folder, "*.json"))
od_pair_geom_data = pd.concat([pd.read_json(f) for f in od_pair_geom_files], ignore_index=False)
od_pair_geom_data.to_json(od_pair_data_geom_path_json, orient="records", default_handler=str, indent=2)


#od_pair_bearings_data.to_csv(od_pair_data_bearings_path_csv)
# Save the concatenated geom data
#od_pair_base_data.to_csv(od_pair_data_base_path_csv)
#od_pair_bearings_data.to_csv(od_pair_data_bearings_path_csv)
#od_pair_geom_data.to_json(od_pair_data_geom_path_json, orient="records", default_handler=str, indent=2)

print("done")


,Unnamed: 0.7,Unnamed: 0.6,Unnamed: 0.5,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,city_name,city_name_en,country_name,country_name_en,continent,region,network_type,node_id,node_latlon,graph_path,weights_added,od_pairs_added
0,0,0,0,0,0,0,0,0,Bangkok,Bangkok,ประเทศไทย,Thailand,Asia,Asia/Oceania,drive,2733298223,"(13.7253693, 100.4953615)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
1,1,1,1,1,1,1,1,1,Bangkok,Bangkok,ประเทศไทย,Thailand,Asia,Asia/Oceania,drive,3825324563,"(13.7843515, 100.4392777)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
2,2,2,2,2,2,2,2,2,Bangkok,Bangkok,ประเทศไทย,Thailand,Asia,Asia/Oceania,drive,1687458348,"(13.7918891, 100.5686954)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
3,3,3,3,3,3,3,3,3,Bangkok,Bangkok,ประเทศไทย,Thailand,Asia,Asia/Oceania,drive,7926838114,"(13.7020039, 100.5380657)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
4,4,4,4,4,4,4,4,4,Bangkok,Bangkok,ประเทศไทย,Thailand,Asia,Asia/Oceania,drive,4772368452,"(13.668279, 100.4044342)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,495,495,495,495,495,495,495,495,Washington,Washington,United States,United States,North America,US/Canada,drive,49896757,"(38.9773557, -77.0191765)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
496,496,496,496,496,496,496,496,496,Washington,Washington,United States,United States,North America,US/Canada,drive,50654959,"(38.8068877, -76.9933723)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
497,497,497,497,497,497,497,497,497,Washington,Washington,United States,United States,North America,US/Canada,drive,49761428,"(38.885008, -76.928403)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True
498,498,498,498,498,498,498,498,498,Washington,Washington,United States,United States,North America,US/Canada,drive,11368228087,"(38.856723, -76.9699374)",/proj/nobackup/streetnetwork-alignment/2025-04...,True,True


odpair data will be stored at /proj/nobackup/streetnetwork-alignment/2025-04-full_dataset/od_pair_data


In [10]:
print(od_pair_base_data['bearings_undirected'])


KeyError: 'bearings_undirected'